## Imports and setup

In [4]:
import matplotlib.pyplot as plt
import networkx as nx
import glob

from networkx.drawing.nx_pydot import graphviz_layout
from tqdm import tqdm

# Ensure plots are displayed inline in Jupyter
%matplotlib inline

## Graph Construction & Visualization Function

In [5]:
def build_kg(dataset_name, context_size, split="train"):
    """
    Constructs the Knowledge Graph using all available data.
    """
    kg = nx.DiGraph()

    # Find all relation CSVs
    file_pattern = (
        f"output/{dataset_name}/{split}/c{context_size}/*_c{context_size}.csv"
    )
    files = glob.glob(file_pattern)

    if not files:
        print(f"No files found for pattern: {file_pattern}")
        return

    print("Scanning files to determine total edge count...")
    total_edges_to_add = 0
    for file_path in files:
        with open(file_path, "r") as f:
            next(f)  # Skip header
            for line in f:
                parts = line.strip().split(";")
                if len(parts) >= 2 and int(parts[1]) == 1:
                    total_edges_to_add += 1

    print(f"Found {total_edges_to_add} valid edges to process.")

    with tqdm(
        total=total_edges_to_add, desc="Building Knowledge Graph", unit=" edges"
    ) as progress_bar:
        for file_path in files:
            with open(file_path, "r") as f:
                next(f)  # Skip header
                for line in f:
                    parts = line.strip().split(";")
                    if len(parts) < 2:
                        continue

                    triple_str = parts[0]
                    label = int(parts[1])

                    if label == 1:
                        try:
                            subject, relation, obj = triple_str.split(",")
                            kg.add_edge(subject, obj, label=relation)
                        except ValueError:
                            continue
                        finally:
                            progress_bar.update(1)

    print(
        f"\nConstructed graph with {kg.number_of_nodes()} nodes and {kg.number_of_edges()} edges."
    )

    return kg


def save_kg(kg, dataset_name, split="train"):
    # Save the full graph data
    graph_filename = f"knowledge_graph_{dataset_name}_{split}_full.graphml"
    nx.write_graphml(kg, graph_filename)
    print(f"Graph data saved locally as: {graph_filename}")


def visualize_kg(kg, dataset_name, split="train"):
    plt.figure(figsize=(20, 15))

    # 'dot' is great for hierarchical/directional flow
    # Alternative programs: 'twopi', 'neato', 'circo'
    pos = graphviz_layout(kg, prog="dot")

    # Draw nodes
    nx.draw_networkx_nodes(
        kg, pos, node_color="skyblue", node_size=1800, alpha=0.9, edgecolors="navy"
    )

    # Draw edges
    nx.draw_networkx_edges(
        kg, pos, edge_color="black", arrows=True, arrowsize=15, width=1.2, alpha=0.6
    )

    # Draw node labels
    nx.draw_networkx_labels(
        kg, pos, font_size=9, font_family="sans-serif", font_weight="bold"
    )

    # Draw edge labels
    edge_labels = nx.get_edge_attributes(kg, "label")
    nx.draw_networkx_edge_labels(
        kg, pos, edge_labels=edge_labels, font_color="darkred", font_size=7
    )

    plt.title(
        f"SciCheck Knowledge Graph: {dataset_name} ({split}) - Graphviz Layout",
        fontsize=18,
    )
    plt.axis("off")
    plt.tight_layout()

    # Save the visualization image
    plot_filename = f"knowledge_graph_{dataset_name}_{split}_graphviz.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    print(f"Graph image saved locally as: {plot_filename}")

    plt.show()

## Execute the visualization

In [2]:
dataset_name = "WN11"
context_size = 3

# Visualizing the ground truth facts from the training set
kg = build_kg(dataset_name, context_size)
save_kg(kg, dataset_name)
visualize_kg(kg, dataset_name)

NameError: name 'build_kg' is not defined